<a href="https://colab.research.google.com/github/anitabudhiraja/DeepLearning/blob/main/LSTM_sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beginner LSTM for Sentiment Analysis
M.TECH AIML

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from collections import Counter
import random

## 1. Small Dataset

In [ ]:
data = [
    ("i love this movie", 1),
    ("this film is great", 1),
    ("i hate this movie", 0),
    ("this film is terrible", 0)
]

random.shuffle(data)

## 2. Tokenization + Vocabulary

In [ ]:
def tokenize(text):
    return text.split()

all_words = []
for text, _ in data:
    all_words.extend(tokenize(text))

vocab = {"<PAD>":0}
for word in set(all_words):
    vocab[word] = len(vocab)

def encode(text):
    return [vocab[w] for w in tokenize(text)]

## 3. Dataset + DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def pad(self, seq, max_len):
        return seq + [0]*(max_len - len(seq))

    def __getitem__(self, idx):
        text, label = self.data[idx]
        encoded = encode(text)
        max_len = 5
        padded = self.pad(encoded, max_len)
        return torch.tensor(padded), torch.tensor(label, dtype=torch.float32)

dataset = TextDataset(data)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

## 4. LSTM Model

In [ ]:
class SimpleLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), 8)
        self.lstm = nn.LSTM(8, 16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return torch.sigmoid(out).squeeze()

model = SimpleLSTM()

## 5. Training

In [ ]:
loss_fn = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(50):
    for X_batch, y_batch in loader:
        preds = model(X_batch)
        loss = loss_fn(preds, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.5274
Epoch 20, Loss: 0.0663
Epoch 30, Loss: 0.0092
Epoch 40, Loss: 0.0036
Epoch 50, Loss: 0.0020


## 6. Testing

In [ ]:
def predict(text):
    encoded = encode(text)
    padded = encoded + [0]*(5-len(encoded))
    x = torch.tensor([padded])
    return model(x).item()

print(predict("i love this movie"))
print(predict("this is terrible"))

0.9987271428108215
0.0023513417690992355


"Model gives a number between 0 and 1"

(1, 0.997)   - Positive

(0, 0.324)   - Negative

In [ ]:
print(predict("i love this"))
print(predict("i hate this"))
print(predict("love love love"))
print(predict("terrible film"))

0.9985156655311584
0.0024033915251493454
0.9985538125038147
0.0025277992244809866
